# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import basic_clean

In [2]:
RAW_PATH = PROJECT_ROOT / "data/raw/raw_dataset.csv"
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(RAW_PATH)
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.lower()
clean_df.head()

,fl_date,airline,airline_dot,airline_code,dot_code,fl_number,origin,origin_city,dest,dest_city,...,diverted,crs_elapsed_time,elapsed_time,air_time,distance,delay_due_carrier,delay_due_weather,delay_due_nas,delay_due_security,delay_due_late_aircraft
0,2021-05-04,JetBlue Airways,JetBlue Airways: B6,B6,20409,384,MCO,"Orlando, FL",JFK,"New York, NY",...,0.0,159.0,152.0,129.0,944.0,NaN,NaN,NaN,NaN,NaN
1,2019-11-26,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,705,FLL,"Fort Lauderdale, FL",DTW,"Detroit, MI",...,0.0,180.0,169.0,150.0,1127.0,NaN,NaN,NaN,NaN,NaN
2,2023-06-18,Southwest Airlines Co.,Southwest Airlines Co.: WN,WN,19393,1926,SMF,"Sacramento, CA",LAS,"Las Vegas, NV",...,0.0,80.0,73.0,59.0,397.0,NaN,NaN,NaN,NaN,NaN
3,2019-07-28,SkyWest Airlines Inc.,SkyWest Airlines Inc.: OO,OO,20304,4459,OKC,"Oklahoma City, OK",DTW,"Detroit, MI",...,0.0,154.0,141.0,122.0,900.0,NaN,NaN,NaN,NaN,NaN
4,2023-03-17,JetBlue Airways,JetBlue Airways: B6,B6,20409,277,FLL,"Fort Lauderdale, FL",SFO,"San Francisco, CA",...,0.0,384.0,402.0,374.0,2584.0,0.0,0.0,18.0,0.0,0.0


### 1. Cancellation Logic

Converted cancellation and diversion flags to binary to enable KPI calculation and ensure uniformity.

In [3]:
clean_df['cancelled'] = clean_df['cancelled'].fillna(0).astype(int)
clean_df['diverted'] = clean_df['diverted'].fillna(0).astype(int)

### 2. Delay Columns

**Missing delay values are assumed as 0 for non-delayed or non-operational flights.**

In [4]:
delay_cols = ['dep_delay', 'arr_delay']

for col in delay_cols:
    clean_df[col] = clean_df[col].fillna(0)

### 3. Delay Cause Columns

**Null delay causes replaced with 0 → indicates no contribution to delay**. These values are naturally missing whenever a delay is under 15 minutes.

In [5]:
delay_causes = [
    'delay_due_carrier',
    'delay_due_weather',
    'delay_due_nas',
    'delay_due_security',
    'delay_due_late_aircraft'
]

for col in delay_causes:
    clean_df[col] = clean_df[col].fillna(0)

### 4. Cancellation Code

Standardized missing cancellation codes for operational flights by explicitly labeling them.

In [6]:
clean_df['cancellation_code'] = clean_df['cancellation_code'].fillna('Not Cancelled')

### 5. Time Columns

Converted HHMM numeric time formats logic into readable standardized proper time formats strings.

In [7]:
import pandas as pd

def convert_time(val):
    if pd.isnull(val):
        return None
    val = int(val)
    hour = val // 100
    minute = val % 100
    return f"{hour:02d}:{minute:02d}"

time_cols = ['dep_time', 'arr_time', 'crs_dep_time', 'crs_arr_time']

for col in time_cols:
    clean_df[col] = clean_df[col].apply(convert_time)

### STEP 3: Feature Engineering

Engineered key analytical features for powerful dashboarding dimensions:
- Delay flags
- Delay buckets
- Route strings
- Year, Month, Day metadata
- Total delay metrics

In [8]:
# 1. Delay Flag
clean_df['is_delayed'] = clean_df['arr_delay'] > 15

# 2. Delay Bucket
def delay_bucket(x):
    if x <= 15:
        return "On Time"
    elif x <= 60:
        return "15-60 min"
    else:
        return "60+ min"

clean_df['delay_bucket'] = clean_df['arr_delay'].apply(delay_bucket)

# 3. Date Features
clean_df['fl_date'] = pd.to_datetime(clean_df['fl_date'])

clean_df['month'] = clean_df['fl_date'].dt.month
clean_df['day_of_week'] = clean_df['fl_date'].dt.day_name()
clean_df['year'] = clean_df['fl_date'].dt.year

# 4. Route Feature
clean_df['route'] = clean_df['origin'] + " → " + clean_df['dest']

# 5. Total Delay Cause
clean_df['total_delay_cause'] = clean_df[delay_causes].sum(axis=1)

### STEP 4: Removing Bad Data

Isolated and removed extreme outlier anomalies to preserve distribution analysis integrity.

In [9]:
clean_df = clean_df[clean_df['arr_delay'] < 500]
clean_df = clean_df[clean_df['dep_delay'] < 500]

clean_df = clean_df.dropna(subset=['airline', 'origin', 'dest'])

In [10]:
# Final processed output generation block
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to {PROCESSED_PATH}')

Saved cleaned dataset to /Users/animeshkumarrai/Desktop/SectionB_G-18_Flight_Delay_Analysis/data/processed/cleaned_dataset.csv


## Cleaning Summary

- Handled missing delay values using domain logic
- Converted cancellation and diversion flags to binary
- Standardized delay cause columns
- Converted time columns into readable format
- Engineered key analytical features:
  - Delay flags
  - Delay buckets
  - Route
  - Month, Year, Day
- Removed outliers and inconsistent records

Final dataset is analysis-ready for KPI computation and Tableau dashboarding.